# PULSO v1 — Entrenamiento y despliegue en Maia

Adaptación del template oficial de Maia al modelo final XGBoost de PULSO. La plataforma inyecta las rutas y el identificador de ejecución mediante Papermill. El notebook **no representa un modelo de riesgo clínico validado**, sino una clasificación educativa de la etiqueta `cardio` del dataset utilizado.

**Flujo:** datos originales/depurados → limpieza reproducible → train/validation/test estratificados (70/15/15) → refit final con train+validation → evaluación test → MLflow → exportación de `model.joblib` con umbral fijo 0.38.

**Contrato:** conservar las etiquetas `parameters`, `mlops:config`, `mlops:data`, `mlops:preprocessing`, `mlops:training`, `mlops:evaluation` y `mlops:export`. Maia debe disponer de `xgboost`, `scikit-learn >=1.5`, `mlflow`, `pyarrow` y `joblib` tanto en el ejecutor como en el servidor de modelos. Verificar compatibilidad exacta de versiones antes de desplegar.

**Datasource de Maia:** seleccionar el CSV original o depurado como entrada; el notebook usa exclusivamente `pd.read_parquet(DATASET_PATH)`, por lo que Maia debe suministrar su versión Parquet. No agregar otro lector de dataset en este cuaderno.

**Interfaz de predicción:** `POST /predict/pulso-xgboost-v1` recibe listas de 10 números **en el orden de `FEATURE_COLUMNS`**. En particular `age_years` y `bmi` deben venir ya calculados con las fórmulas documentadas. El modelo exportado incluye el preprocesador categórico y el umbral; **no** calcula estas derivadas a partir de las otras ocho entradas.


In [ ]:
# ================================================================
# 🔒 NO TOCAR — celda "parameters" (etiqueta: parameters)
# ================================================================
# Papermill inyecta una celda NUEVA justo después de esta con los valores reales.
# Aquí solo se declaran con un valor vacío para que el notebook corra también a mano.
DATASET_PATH = ""            # Archivo local con los datos elegidos en el nodo Input
MODEL_OUTPUT_PATH = ""       # Dónde guardar el modelo entrenado (.joblib)
PIPELINE_ID = ""             # UUID de la ejecución
MLFLOW_TRACKING_URI = ""     # URL del servidor MLflow
MLFLOW_RUN_ID = ""           # Run de MLflow donde DEBES registrar las métricas
TRAINING_FEATURES_PATH = ""  # Dónde escribir el DataFrame exacto pasado a fit()
MODEL_NOTES_PATH = ""        # Dónde escribir qué significa cada entrada / clase

# ================================================================
# ✏️ TU PARTE — hiperparámetros (opcional)
# ================================================================
# Declara aquí, con un valor por defecto, los parámetros que quieras cambiar desde
# "Run pipeline" sin editar el código. Tienen que estar EN ESTA CELDA: si los defines
# más abajo, tu asignación pisa el valor que inyecta Maia.
# Nombres válidos: identificadores Python, máximo 30, sin usar los nombres de arriba.
# Ejemplo:
# N_ESTIMATORS = 200
# MAX_DEPTH = 8
# TEST_SIZE = 0.2

In [ ]:
# ================================================================
# mlops:config — identificación del modelo
# ================================================================
import os

MODEL_NAME = "pulso-xgboost-v1"
VERSION = "1"
MODEL_OUTPUT_PATH = MODEL_OUTPUT_PATH or "./model.joblib"
PIPELINE_ID = PIPELINE_ID or "local-dev"
MLFLOW_TRACKING_URI = MLFLOW_TRACKING_URI or os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")

# Configuración congelada de PULSO v1 (final_model.yaml)
THRESHOLD = 0.38
RANDOM_STATE = 42
MODEL_PARAMS = {
    "colsample_bytree": 0.8,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 5,
    "n_estimators": 300,
    "reg_lambda": 5,
    "subsample": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}
print(f"Modelo: {MODEL_NAME} v{VERSION} | datos: {DATASET_PATH or '(ninguno)'}")


In [ ]:
# ================================================================
# mlops:data — dataset suministrado por Maia
# ================================================================
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

TARGET_COLUMN = "cardio"
if not DATASET_PATH or not Path(DATASET_PATH).is_file():
    raise ValueError("DATASET_PATH no existe. Selecciona un datasource en el nodo Input de Maia.")
LAYER = Path(DATASET_PATH).name.split("-")[0]
df = pd.read_parquet(DATASET_PATH)
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"La variable objetivo {TARGET_COLUMN!r} no está presente en el dataset.")
DATASET_SHA256 = hashlib.sha256(Path(DATASET_PATH).read_bytes()).hexdigest()
print(f"Capa: {LAYER} | registros originales recibidos: {len(df):,} | columnas: {len(df.columns)}")


In [ ]:
# ================================================================
# mlops:preprocessing — mismo contrato de ml/prepare.py y ml/common.py
# ================================================================
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

FEATURE_COLUMNS = [
    "gender", "height", "weight", "ap_hi", "ap_lo",
    "smoke", "alco", "active", "age_years", "bmi",
]
CONTINUOUS = ["height", "weight", "ap_hi", "ap_lo", "age_years", "bmi"]
CATEGORICAL = ["gender"]
BINARY = ["smoke", "alco", "active"]
REQUIRED_BASE = ["age", "gender", "height", "weight", "ap_hi", "ap_lo", "smoke", "alco", "active", TARGET_COLUMN]
missing = sorted(set(REQUIRED_BASE) - set(df.columns))
if missing:
    raise ValueError(f"Faltan columnas requeridas para reproducir la preparación: {missing}")

# Aplicación idéntica de reglas: también acepta CSV depurado convertido a Parquet.
valid_mask = (
    df["height"].between(120, 220)
    & df["weight"].between(30, 200)
    & df["ap_hi"].between(70, 250)
    & df["ap_lo"].between(40, 150)
    & (df["ap_hi"] > df["ap_lo"])
)
data = df.loc[valid_mask].copy()
data["age_years"] = data["age"] / 365.25
data["bmi"] = data["weight"] / ((data["height"] / 100) ** 2)
if data.empty:
    raise ValueError("La limpieza dejó cero registros.")
if data[FEATURE_COLUMNS + [TARGET_COLUMN]].isna().any().any():
    raise ValueError("Datos faltantes en los predictores o la etiqueta")
if not np.isfinite(data[FEATURE_COLUMNS + [TARGET_COLUMN]].to_numpy(dtype=float)).all():
    raise ValueError("Valores no finitos en datos de entrenamiento")
if not all(pd.api.types.is_numeric_dtype(data[c]) for c in FEATURE_COLUMNS + [TARGET_COLUMN]):
    raise ValueError("Maia requiere variables numéricas")
if set(data[TARGET_COLUMN].unique()) != {0, 1}:
    raise ValueError("cardio debe contener exactamente las clases 0 y 1")

X = data[FEATURE_COLUMNS].copy()
y = data[TARGET_COLUMN].copy()
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)
# Refit final, como en ml/final_evaluate.py
X_final_train = pd.concat([X_train, X_val], axis=0)
y_final_train = pd.concat([y_train, y_val], axis=0)
print(f"Filas recibidas: {len(df):,} | depuradas: {len(data):,} | train: {len(X_train):,} | val: {len(X_val):,} | test: {len(X_test):,}")


In [ ]:
# ================================================================
# mlops:training — pipeline XGBoost oficial + umbral fijo desplegable
# ================================================================
from xgboost import XGBClassifier
from sklearn.model_selection import FixedThresholdClassifier

preprocessor = ColumnTransformer(transformers=[
    ("continuous", "passthrough", [FEATURE_COLUMNS.index(c) for c in CONTINUOUS]),
    ("gender", OneHotEncoder(drop=None, handle_unknown="ignore"), [FEATURE_COLUMNS.index(c) for c in CATEGORICAL]),
    ("binary", "passthrough", [FEATURE_COLUMNS.index(c) for c in BINARY]),
])
base_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(**MODEL_PARAMS)),
])
# ColumnTransformer usa índices posicionales: Maia envía matrices NumPy, no DataFrames.
# FixedThresholdClassifier es un objeto nativo de sklearn: joblib puede
# cargarlo sin depender de clases Python definidas en este notebook.
# Entrena el pipeline interno con el conjunto final (train + validation).
model = FixedThresholdClassifier(
    estimator=base_pipeline,
    threshold=THRESHOLD,
    response_method="predict_proba",
)
model.fit(X_final_train, y_final_train)
print("Entrenamiento final completado. El predict() desplegado utiliza threshold=0.38.")


In [ ]:
# ================================================================
# mlops:evaluation — solo test reservado, MLflow administrado por Maia
# ================================================================
import mlflow
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, brier_score_loss, log_loss,
)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

# Misma definición del ECE oficial: bins uniformes, como ml/calibration.py.
# Verificar esta definición contra ml/calibration.py antes de comparar el ECE.
def ece_uniform(y_true, proba, n_bins=10):
    yt = np.asarray(y_true)
    pp = np.asarray(proba)
    bounds = np.linspace(0, 1, n_bins + 1)
    bins = np.clip(np.digitize(pp, bounds[1:-1], right=False), 0, n_bins - 1)
    return float(sum((np.sum(bins == i) / len(pp)) * abs(np.mean(yt[bins == i]) - np.mean(pp[bins == i]))
                     for i in range(n_bins) if np.any(bins == i)))

METRICS = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
    "brier": float(brier_score_loss(y_test, y_proba)),
    "log_loss": float(log_loss(y_test, y_proba)),
    "ece": ece_uniform(y_test, y_proba),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
}
PARAMS = {
    "model_type": "xgboost", "threshold": THRESHOLD,
    "calibration": "none", "refit_strategy": "train_plus_validation",
    "dataset_sha256": DATASET_SHA256, "random_state": RANDOM_STATE,
    "final_train_rows": len(X_final_train), "test_rows": len(X_test),
    "layer": LAYER, "target_column": TARGET_COLUMN,
    "features": ",".join(FEATURE_COLUMNS),
    **{f"model__{key}": value for key, value in MODEL_PARAMS.items()},
}
# El run pertenece a Maia. No abrir otro experimento separado.
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
started_here = mlflow.active_run() is None
if started_here:
    if MLFLOW_RUN_ID:
        mlflow.start_run(run_id=MLFLOW_RUN_ID)
    else:
        mlflow.start_run(run_name=f"{MODEL_NAME}-{PIPELINE_ID}")
mlflow.log_params(PARAMS)
mlflow.log_metrics({name: float(value) for name, value in METRICS.items()})
mlflow.log_metric("n_samples", float(len(X)))
if started_here:
    mlflow.end_run()
print("Test:", {name: round(float(value), 6) for name, value in METRICS.items()})


In [ ]:
# ================================================================
# mlops:export — contrato del servidor de Maia
# ================================================================
import json
import joblib

# Comprobar que el artefacto serializado aplica el umbral 0.38.
assert np.array_equal(model.predict(X_test.iloc[:32]), (model.predict_proba(X_test.iloc[:32])[:, 1] >= THRESHOLD).astype(int))
Path(MODEL_OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_OUTPUT_PATH)
reloaded = joblib.load(MODEL_OUTPUT_PATH)
assert np.array_equal(reloaded.predict(X_test.iloc[:32]), model.predict(X_test.iloc[:32]))
assert np.array_equal(reloaded.predict(X_test.iloc[:32].to_numpy()), model.predict(X_test.iloc[:32]))
print(f"Modelo guardado en {MODEL_OUTPUT_PATH}")

# Maia construye su formulario a partir de las entradas usadas en fit().
if TRAINING_FEATURES_PATH:
    Path(TRAINING_FEATURES_PATH).parent.mkdir(parents=True, exist_ok=True)
    X_final_train.to_parquet(TRAINING_FEATURES_PATH)

NOTES = {
    "target": {"name": "cardio", "description": "Etiqueta binaria del dataset cardiovascular (no riesgo clínico absoluto)."},
    "inputs": {
        "gender": {"description": "Sexo codificado según la fuente (1 o 2)."},
        "height": {"description": "Estatura.", "unit": "cm"},
        "weight": {"description": "Peso.", "unit": "kg"},
        "ap_hi": {"description": "Presión arterial sistólica.", "unit": "mmHg"},
        "ap_lo": {"description": "Presión arterial diastólica.", "unit": "mmHg"},
        "smoke": {"description": "Tabaquismo reportado: 0 no, 1 sí."},
        "alco": {"description": "Consumo de alcohol reportado: 0 no, 1 sí."},
        "active": {"description": "Actividad física reportada: 0 no, 1 sí."},
        "age_years": {"description": "Edad en años = edad original en días / 365.25.", "unit": "años"},
        "bmi": {"description": "Índice de masa corporal = peso(kg) / estatura(m)^2.", "unit": "kg/m²"},
    },
    "classes": {
        "0": "Registro sin la condición codificada en cardio",
        "1": "Registro con la condición codificada en cardio",
    },
}
if MODEL_NOTES_PATH:
    Path(MODEL_NOTES_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(MODEL_NOTES_PATH, "w", encoding="utf-8") as handle:
        json.dump(NOTES, handle, ensure_ascii=False, indent=2)


## Reglas de integración y límites

- **Política de Maia:** seleccionar `recall` como métrica de despliegue (dirección *higher*). Considerar que cumplir recall no demuestra utilidad clínica ni garantiza especificidad adecuada. La política debe configurarse en Maia, no en el notebook.
- **Versiones:** `FixedThresholdClassifier` está disponible desde scikit-learn 1.5; el servicio que carga `model.joblib` necesita scikit-learn, XGBoost y las dependencias compatibles. No desplegar sin comprobarlo.
- **Fuente:** usar el dataset original de 70.000 registros o el depurado de 68.610. El notebook reaplica las reglas documentadas, por lo que ambos deberían dar 68.610 observaciones si no hubo transformaciones adicionales de Maia. Si los números difieren, investigar antes de publicar métricas.
- **Modelo sin interfaz clínica:** la predicción recibe diez números, incluso `age_years` y `bmi`; a diferencia de la web PULSO, no dispone del backend que calcula y valida estos valores.
- **Control de versiones:** comparar resultados de test con `artifacts/final/test_metrics.csv` antes de declarar equivalencia con PULSO v1. El notebook no carga el artefacto original: crea uno nuevo y comparable.
